In [1]:
# ===== 1. 설정 =====
import os
from pathlib import Path

# 저장 위치: <USER_HOME>\OneDrive\INVESTMENT\지표상회_복제\result_files\KOSIS
USER_HOME = Path(r"C:\Users\82108")
SUBPATH = Path("OneDrive") / "INVESTMENT" / "지표상회_복제" / "result_files" / "KOSIS"


def kosis_output_dir():
    """저장 폴더 경로를 결정한다.

    1) 환경변수 KOSIS_OUTPUT_DIR 이 있으면 최우선
    2) USER_HOME 이 실제로 있으면 USER_HOME / SUBPATH
    3) 없으면 현재 사용자 홈 아래에 같은 구조로 저장
       (3번 분기가 없으면 PermissionError WinError 5 가 발생한다)
    """
    env = os.environ.get("KOSIS_OUTPUT_DIR")
    if env:
        return Path(env)

    if USER_HOME.is_dir():
        return USER_HOME / SUBPATH

    fallback = Path.home() / SUBPATH
    print(f"[안내] {USER_HOME} 를 찾을 수 없습니다. {Path.home()} 기준으로 저장합니다.")
    return fallback


SAVE_DIR = kosis_output_dir()
SAVE_DIR.mkdir(parents=True, exist_ok=True)   # KOSIS 폴더까지 자동 생성

# API 키: 노트북에 남기고 싶지 않으면 환경변수 KOSIS_API_KEY 를 대신 설정
KOSIS_API_KEY = os.environ.get(
    "KOSIS_API_KEY",
    "ZTFhMjg1MzhmNmFiYWJlYmY3ZWUxZDA0ZDI2ZTM0YWU="
)

print("저장 폴더 :", SAVE_DIR)
print("폴더 존재 :", SAVE_DIR.is_dir())


저장 폴더 : C:\Users\82108\OneDrive\INVESTMENT\지표상회_복제\result_files\KOSIS
폴더 존재 : True


In [2]:
# ===== 2. 수집기 정의 =====
import requests
import pandas as pd
from datetime import datetime
import logging
from typing import Optional, Dict, List
import json
import os

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

class KOSISRetailCollector:
    """KOSIS 소매판매 데이터 수집 (판매채널별 + 제화별 + 온라인)"""

    def __init__(self, api_key: str, save_dir: str = None):
        self.api_key = api_key
        self.base_url = "https://kosis.kr/openapi/Param/statisticsParameterData.do"

        # 저장 디렉토리 설정 (설정 셀의 SAVE_DIR을 기본값으로 사용)
        if save_dir is None:
            save_dir = SAVE_DIR
        self.save_dir = str(save_dir)

        # 디렉토리가 없으면 생성
        os.makedirs(self.save_dir, exist_ok=True)
        logger.info(f"저장 디렉토리: {self.save_dir}")

    def get_data_by_table(self, table_id: str, item_id: str = 'T1',
                         obj_l1: str = 'ALL', obj_l2: str = '',
                         months: int = 24) -> Optional[pd.DataFrame]:
        """
        KOSIS 통계표별 데이터 수집

        Parameters:
        -----------
        table_id : str
            통계표 ID (예: DT_1K41002, DT_1K41003, DT_1KE10041)
        item_id : str
            항목 ID (기본값: 'T1')
        obj_l1 : str
            객체 레벨1 (기본값: 'ALL')
        obj_l2 : str
            객체 레벨2 (기본값: '')
        months : int
            최근 몇 개월 데이터 (기본값: 24개월)
        """
        params = {
            'method': 'getList',
            'apiKey': self.api_key,
            'itmId': item_id,
            'objL1': obj_l1,
            'objL2': obj_l2,
            'objL3': '',
            'objL4': '',
            'objL5': '',
            'objL6': '',
            'objL7': '',
            'objL8': '',
            'format': 'json',
            'jsonVD': 'Y',
            'prdSe': 'M',  # 월별
            'newEstPrdCnt': str(months),
            'orgId': '101',
            'tblId': table_id
        }

        try:
            logger.info(f"KOSIS 데이터 요청 (테이블: {table_id}, 최근 {months}개월)")
            response = requests.get(self.base_url, params=params, timeout=30)
            response.raise_for_status()

            data = response.json()

            if not data:
                logger.warning(f"응답 데이터가 비어있습니다 (테이블: {table_id})")
                return None

            df = pd.DataFrame(data)
            logger.info(f"데이터 수집 완료 (테이블: {table_id}): {len(df)} rows")
            logger.info(f"컬럼: {df.columns.tolist()}")

            # 테이블 ID 추가
            df['source_table'] = table_id

            return df

        except requests.exceptions.RequestException as e:
            logger.error(f"API 요청 실패 (테이블: {table_id}): {e}")
            return None
        except Exception as e:
            logger.error(f"데이터 처리 중 오류 (테이블: {table_id}): {e}")
            return None

    def get_channel_sales(self, months: int = 24) -> Optional[pd.DataFrame]:
        """판매채널별 판매액 데이터 수집 (DT_1K41003)"""
        return self.get_data_by_table('DT_1K41003', item_id='T1', months=months)

    def get_product_sales(self, months: int = 24) -> Optional[pd.DataFrame]:
        """제화별 판매액 데이터 수집 (DT_1K41002)"""
        return self.get_data_by_table('DT_1K41002', item_id='T1', months=months)

    def get_online_sales(self, months: int = 24) -> Optional[pd.DataFrame]:
        """온라인 소매판매액 데이터 수집 (DT_1KE10041)"""
        return self.get_data_by_table('DT_1KE10041', item_id='T20', obj_l1='ALL', obj_l2='ALL', months=months)

    def process_data(self, df: pd.DataFrame, data_type: str = 'auto') -> pd.DataFrame:
        """
        데이터 전처리 및 정제

        Parameters:
        -----------
        data_type : str
            'channel' (판매채널별), 'product' (제화별), 'online' (온라인), 'auto' (자동감지)
        """
        if df is None or df.empty:
            logger.warning("처리할 데이터가 없습니다")
            return pd.DataFrame()

        logger.info(f"데이터 전처리 시작 (타입: {data_type})")
        processed = df.copy()

        # 데이터 타입 자동 감지
        if data_type == 'auto':
            if 'source_table' in processed.columns:
                if processed['source_table'].iloc[0] == 'DT_1K41003':
                    data_type = 'channel'
                elif processed['source_table'].iloc[0] == 'DT_1K41002':
                    data_type = 'product'
                elif processed['source_table'].iloc[0] == 'DT_1KE10041':
                    data_type = 'online'

        processed['data_type'] = data_type

        # 1. 날짜 처리
        if 'PRD_DE' in processed.columns:
            processed['date'] = pd.to_datetime(
                processed['PRD_DE'],
                format='%Y%m',
                errors='coerce'
            )
            processed['year'] = processed['date'].dt.year
            processed['month'] = processed['date'].dt.month
            processed['year_month'] = processed['PRD_DE']

        # 2. 카테고리 정보 (판매채널, 제화, 온라인 상품군)
        category_cols = ['C1_NM', 'C2_NM', 'C1', 'C2']
        for col in category_cols:
            if col in processed.columns:
                processed[f'category_{col.lower()}'] = processed[col]

        # 3. 판매액 (숫자로 변환)
        if 'DT' in processed.columns:
            processed['sales_amount'] = pd.to_numeric(
                processed['DT'].astype(str).str.replace(',', '').str.strip(),
                errors='coerce'
            )

        # 4. 단위 정보
        if 'UNIT_NM' in processed.columns:
            processed['unit'] = processed['UNIT_NM']

        # 5. 항목명
        if 'ITM_NM' in processed.columns:
            processed['item_name'] = processed['ITM_NM']

        # 6. 통계표 정보
        if 'TBL_NM' in processed.columns:
            processed['table_name'] = processed['TBL_NM']

        if 'TBL_ID' in processed.columns:
            processed['table_id'] = processed['TBL_ID']

        if 'ORG_ID' in processed.columns:
            processed['org_id'] = processed['ORG_ID']

        # 7. 데이터 수집 시간
        processed['collected_at'] = datetime.now()

        # 8. 결측치 확인
        logger.info(f"결측치 확인:\n{processed.isnull().sum()}")

        return processed

    def create_summary(self, df: pd.DataFrame) -> pd.DataFrame:
        """카테고리별 요약 통계"""
        if df.empty or 'sales_amount' not in df.columns:
            logger.warning("요약 통계를 생성할 수 없습니다")
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None:
            logger.warning("카테고리 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        summary = df.groupby(category_col).agg({
            'sales_amount': ['count', 'mean', 'sum', 'min', 'max', 'std'],
            'date': ['min', 'max']
        }).round(2)

        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        summary = summary.reset_index()
        summary.columns = [
            'category', 'record_count', 'avg_sales', 'total_sales',
            'min_sales', 'max_sales', 'std_sales', 'first_date', 'last_date'
        ]

        summary = summary.sort_values('total_sales', ascending=False)

        return summary

    def create_pivot_table(self, df: pd.DataFrame) -> pd.DataFrame:
        """날짜 x 카테고리 피벗 테이블 생성"""
        if df.empty:
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None or 'date' not in df.columns:
            logger.warning("피벗 테이블을 생성할 수 없습니다")
            return pd.DataFrame()

        pivot = df.pivot_table(
            values='sales_amount',
            index='date',
            columns=category_col,
            aggfunc='sum'
        )

        pivot = pivot.sort_index()

        return pivot

    def calculate_growth_rate(self, df: pd.DataFrame) -> pd.DataFrame:
        """전년 동월 대비 성장률 계산"""
        if df.empty or 'date' not in df.columns:
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None:
            return pd.DataFrame()

        df_sorted = df.sort_values(['date', category_col]).copy()

        # 전년 동월 대비 성장률
        df_sorted['yoy_growth'] = df_sorted.groupby(category_col)['sales_amount'].pct_change(12) * 100

        # 전월 대비 성장률
        df_sorted['mom_growth'] = df_sorted.groupby(category_col)['sales_amount'].pct_change(1) * 100

        return df_sorted

    def create_yoy_growth_pivot(self, df: pd.DataFrame) -> pd.DataFrame:
        """전년 동월 대비 성장률(YoY Growth) 피벗 테이블 생성"""
        if df.empty:
            logger.warning("데이터가 비어있습니다")
            return pd.DataFrame()

        if 'yoy_growth' not in df.columns:
            logger.warning("yoy_growth 컬럼이 없습니다. calculate_growth_rate()를 먼저 실행하세요")
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None or 'date' not in df.columns:
            logger.warning("날짜 또는 카테고리 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        # YoY 성장률 피벗 테이블 생성
        yoy_pivot = df.pivot_table(
            values='yoy_growth',
            index='date',
            columns=category_col,
            aggfunc='mean'
        )

        yoy_pivot = yoy_pivot.sort_index()
        yoy_pivot = yoy_pivot.round(2)

        logger.info(f"YoY 성장률 피벗 테이블 생성 완료: {yoy_pivot.shape}")

        return yoy_pivot

    def create_mom_growth_pivot(self, df: pd.DataFrame) -> pd.DataFrame:
        """전월 대비 성장률(MoM Growth) 피벗 테이블 생성"""
        if df.empty:
            logger.warning("데이터가 비어있습니다")
            return pd.DataFrame()

        if 'mom_growth' not in df.columns:
            logger.warning("mom_growth 컬럼이 없습니다. calculate_growth_rate()를 먼저 실행하세요")
            return pd.DataFrame()

        # 카테고리 컬럼 찾기
        category_col = None
        for col in ['category_c1_nm', 'category_c2_nm', 'C1_NM', 'C2_NM']:
            if col in df.columns:
                category_col = col
                break

        if category_col is None or 'date' not in df.columns:
            logger.warning("날짜 또는 카테고리 정보를 찾을 수 없습니다")
            return pd.DataFrame()

        # MoM 성장률 피벗 테이블 생성
        mom_pivot = df.pivot_table(
            values='mom_growth',
            index='date',
            columns=category_col,
            aggfunc='mean'
        )

        mom_pivot = mom_pivot.sort_index()
        mom_pivot = mom_pivot.round(2)

        logger.info(f"MoM 성장률 피벗 테이블 생성 완료: {mom_pivot.shape}")

        return mom_pivot

    def get_filename_with_date(self, base_name: str = 'kosis_retail_sales') -> str:
        """날짜가 포함된 파일명 생성"""
        today = datetime.now().strftime('%Y%m%d')
        filename = f"{base_name}_{today}.xlsx"
        filepath = os.path.join(self.save_dir, filename)
        return filepath

    def export_to_excel(self, results_dict: Dict[str, Dict[str, pd.DataFrame]],
                       filename: str = None):
        """여러 데이터셋을 하나의 Excel 파일로 저장"""

        # 파일명 생성 (날짜 포함)
        if filename is None:
            filepath = self.get_filename_with_date()
        else:
            filepath = os.path.join(self.save_dir, filename)

        try:
            with pd.ExcelWriter(filepath, engine='openpyxl') as writer:

                # 판매채널별 데이터
                if 'channel' in results_dict:
                    channel = results_dict['channel']

                    if not channel.get('processed', pd.DataFrame()).empty:
                        channel['processed'].to_excel(writer, sheet_name='채널_원본데이터', index=False)

                    if not channel.get('summary', pd.DataFrame()).empty:
                        channel['summary'].to_excel(writer, sheet_name='채널_요약', index=False)

                    if not channel.get('pivot', pd.DataFrame()).empty:
                        channel['pivot'].to_excel(writer, sheet_name='채널_판매액_피벗')

                    if not channel.get('growth', pd.DataFrame()).empty:
                        channel['growth'].to_excel(writer, sheet_name='채널_성장률분석', index=False)

                    if not channel.get('yoy_pivot', pd.DataFrame()).empty:
                        channel['yoy_pivot'].to_excel(writer, sheet_name='채널_YoY_피벗')

                    if not channel.get('mom_pivot', pd.DataFrame()).empty:
                        channel['mom_pivot'].to_excel(writer, sheet_name='채널_MoM_피벗')

                # 제화별 데이터
                if 'product' in results_dict:
                    product = results_dict['product']

                    if not product.get('processed', pd.DataFrame()).empty:
                        product['processed'].to_excel(writer, sheet_name='제화_원본데이터', index=False)

                    if not product.get('summary', pd.DataFrame()).empty:
                        product['summary'].to_excel(writer, sheet_name='제화_요약', index=False)

                    if not product.get('pivot', pd.DataFrame()).empty:
                        product['pivot'].to_excel(writer, sheet_name='제화_판매액_피벗')

                    if not product.get('growth', pd.DataFrame()).empty:
                        product['growth'].to_excel(writer, sheet_name='제화_성장률분석', index=False)

                    if not product.get('yoy_pivot', pd.DataFrame()).empty:
                        product['yoy_pivot'].to_excel(writer, sheet_name='제화_YoY_피벗')

                    if not product.get('mom_pivot', pd.DataFrame()).empty:
                        product['mom_pivot'].to_excel(writer, sheet_name='제화_MoM_피벗')

                # 온라인 판매 데이터
                if 'online' in results_dict:
                    online = results_dict['online']

                    if not online.get('processed', pd.DataFrame()).empty:
                        online['processed'].to_excel(writer, sheet_name='온라인_원본데이터', index=False)

                    if not online.get('summary', pd.DataFrame()).empty:
                        online['summary'].to_excel(writer, sheet_name='온라인_요약', index=False)

                    if not online.get('pivot', pd.DataFrame()).empty:
                        online['pivot'].to_excel(writer, sheet_name='온라인_판매액_피벗')

                    if not online.get('growth', pd.DataFrame()).empty:
                        online['growth'].to_excel(writer, sheet_name='온라인_성장률분석', index=False)

                    if not online.get('yoy_pivot', pd.DataFrame()).empty:
                        online['yoy_pivot'].to_excel(writer, sheet_name='온라인_YoY_피벗')

                    if not online.get('mom_pivot', pd.DataFrame()).empty:
                        online['mom_pivot'].to_excel(writer, sheet_name='온라인_MoM_피벗')

            logger.info(f"Excel 파일 저장 완료: {filepath}")

        except Exception as e:
            logger.error(f"Excel 저장 실패: {e}")

    def run_single_collection(self, table_id: str, data_type: str,
                             item_id: str = 'T1', obj_l1: str = 'ALL',
                             obj_l2: str = '', months: int = 24) -> Dict[str, pd.DataFrame]:
        """단일 테이블 수집 프로세스"""
        logger.info(f"=== {data_type} 데이터 수집 시작 (테이블: {table_id}) ===")

        # 1. 데이터 수집
        raw_data = self.get_data_by_table(table_id, item_id, obj_l1, obj_l2, months)
        if raw_data is None or raw_data.empty:
            logger.error(f"{data_type} 데이터 수집 실패")
            return {
                'raw': pd.DataFrame(),
                'processed': pd.DataFrame(),
                'summary': pd.DataFrame(),
                'pivot': pd.DataFrame(),
                'growth': pd.DataFrame(),
                'yoy_pivot': pd.DataFrame(),
                'mom_pivot': pd.DataFrame()
            }

        # 2. 데이터 처리
        processed_data = self.process_data(raw_data, data_type)

        # 3. 요약 통계 생성
        summary = self.create_summary(processed_data)
        if not summary.empty:
            logger.info(f"\n=== {data_type} 요약 통계 ===")
            print(summary.to_string(index=False))

        # 4. 판매액 피벗 테이블 생성
        pivot = self.create_pivot_table(processed_data)
        if not pivot.empty:
            logger.info(f"\n=== {data_type} 판매액 피벗 테이블 ===")
            print(pivot.tail(10))

        # 5. 성장률 계산
        growth_data = self.calculate_growth_rate(processed_data)

        # 6. YoY 성장률 피벗 테이블 생성
        yoy_pivot = self.create_yoy_growth_pivot(growth_data)
        if not yoy_pivot.empty:
            logger.info(f"\n=== {data_type} YoY 성장률 피벗 테이블 ===")
            print(yoy_pivot.tail(12))

        # 7. MoM 성장률 피벗 테이블 생성
        mom_pivot = self.create_mom_growth_pivot(growth_data)

        logger.info(f"=== {data_type} 데이터 수집 완료 ===\n")

        return {
            'raw': raw_data,
            'processed': processed_data,
            'summary': summary,
            'pivot': pivot,
            'growth': growth_data,
            'yoy_pivot': yoy_pivot,
            'mom_pivot': mom_pivot
        }

    def run_all_collection(self, months: int = 24, save_excel: bool = True) -> Dict[str, Dict[str, pd.DataFrame]]:
        """
        판매채널별 + 제화별 + 온라인 데이터 모두 수집

        Returns:
        --------
        dict: 'channel', 'product', 'online' 키를 가진 딕셔너리
        """
        logger.info("=== KOSIS 소매판매 전체 데이터 수집 시작 ===\n")

        results = {}

        # 1. 판매채널별 데이터 수집 (DT_1K41003)
        results['channel'] = self.run_single_collection(
            table_id='DT_1K41003',
            data_type='channel',
            item_id='T1',
            obj_l1='ALL',
            obj_l2='',
            months=months
        )

        # 2. 제화별 데이터 수집 (DT_1K41002)
        results['product'] = self.run_single_collection(
            table_id='DT_1K41002',
            data_type='product',
            item_id='T1',
            obj_l1='ALL',
            obj_l2='',
            months=months
        )

        # 3. 온라인 소매판매 데이터 수집 (DT_1KE10041)
        results['online'] = self.run_single_collection(
            table_id='DT_1KE10041',
            data_type='online',
            item_id='T20',
            obj_l1='ALL',
            obj_l2='ALL',
            months=months
        )

        # 4. Excel 저장
        if save_excel:
            self.export_to_excel(results)

        logger.info("=== KOSIS 소매판매 전체 데이터 수집 완료 ===")

        return results


In [3]:
# ===== 3. 실행 =====
# months: 최근 몇 개월치를 받을지 (36 = 3년)
collector = KOSISRetailCollector(KOSIS_API_KEY, save_dir=SAVE_DIR)
all_results = collector.run_all_collection(months=36, save_excel=True)

channel_results = all_results['channel']
product_results = all_results['product']
online_results  = all_results['online']

for label, res in [("판매채널별", channel_results),
                   ("제화별", product_results),
                   ("온라인", online_results)]:
    print(f"\n=== {label} YoY 성장률 (최근 6개월) ===")
    if not res['yoy_pivot'].empty:
        print(res['yoy_pivot'].tail(6))
    else:
        print("(데이터 없음)")

print(f"\n저장 위치: {SAVE_DIR}")
print(f"파일명   : kosis_retail_sales_YYYYMMDD.xlsx")


2026-07-27 21:46:18,930 - INFO - 저장 디렉토리: C:\Users\82108\OneDrive\INVESTMENT\지표상회_복제\result_files\KOSIS
2026-07-27 21:46:18,931 - INFO - === KOSIS 소매판매 전체 데이터 수집 시작 ===

2026-07-27 21:46:18,931 - INFO - === channel 데이터 수집 시작 (테이블: DT_1K41003) ===
2026-07-27 21:46:18,932 - INFO - KOSIS 데이터 요청 (테이블: DT_1K41003, 최근 36개월)
2026-07-27 21:46:19,257 - INFO - 데이터 수집 완료 (테이블: DT_1K41003): 288 rows
2026-07-27 21:46:19,258 - INFO - 컬럼: ['C1_OBJ_NM', 'DT', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG']
2026-07-27 21:46:19,260 - INFO - 데이터 전처리 시작 (타입: channel)
2026-07-27 21:46:19,270 - INFO - 결측치 확인:
C1_OBJ_NM         0
DT                0
C1                0
PRD_SE            0
UNIT_NM_ENG       0
ITM_ID            0
TBL_ID            0
ITM_NM            0
TBL_NM            0
PRD_DE            0
LST_CHN_DE        0
C1_NM_ENG         0
C1_NM             0
UNIT_NM           0
ITM_

    category  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
       전문소매점            36 15723949.06    566062166   14369072   17156831  714410.59 2023-06-01 2026-05-01
      무점포 소매            36 11705598.92    421401561   10688426   13194616  626670.01 2023-06-01 2026-05-01
승용차 및 연료 소매점            36 10959637.78    394546960    9274173   13103237  785667.51 2023-06-01 2026-05-01
  슈퍼마켓 및 잡화점            36  5641261.89    203085428    4836426    6411664  277602.67 2023-06-01 2026-05-01
         백화점            36  3474146.97    125069291    2967662    4264516  324025.35 2023-06-01 2026-05-01
        대형마트            36  3055255.36    109989193    2597079    3862486  247909.15 2023-06-01 2026-05-01
         편의점            36  2637838.75     94962195    2180162    2888716  178738.14 2023-06-01 2026-05-01
         면세점            36  1126014.28     40536514     915232    1590894  136384.53 2023-06-01 2026-05-01
category_c1_nm     대형마트      면세점    무

2026-07-27 21:46:19,665 - INFO - 데이터 수집 완료 (테이블: DT_1K41002): 720 rows
2026-07-27 21:46:19,666 - INFO - 컬럼: ['C1_OBJ_NM', 'DT', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG']
2026-07-27 21:46:19,669 - INFO - 데이터 전처리 시작 (타입: product)
2026-07-27 21:46:19,679 - INFO - 결측치 확인:
C1_OBJ_NM         0
DT                0
C1                0
PRD_SE            0
UNIT_NM_ENG       0
ITM_ID            0
TBL_ID            0
ITM_NM            0
TBL_NM            0
PRD_DE            0
LST_CHN_DE        0
C1_NM_ENG         0
C1_NM             0
UNIT_NM           0
ITM_NM_ENG        0
ORG_ID            0
C1_OBJ_NM_ENG     0
source_table      0
data_type         0
date              0
year              0
month             0
year_month        0
category_c1_nm    0
category_c1       0
sales_amount      0
unit              0
item_name         0
table_name        0
table_id          0
org_

  category  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
        합계            36 54323703.00   1955653308   49774993   59170243 2096188.55 2023-06-01 2026-05-01
합계(승용차 제외)            36 48750039.36   1755001417   44466190   52844622 1782333.74 2023-06-01 2026-05-01
      비내구재            36 30230373.47   1088293445   27386732   32769820 1246129.12 2023-06-01 2026-05-01
      음식료품            36 15220752.31    547947083   13181558   17766442  972427.27 2023-06-01 2026-05-01
       내구재            36 13527388.00    486985968   11506876   16263408  827590.97 2023-06-01 2026-05-01
      준내구재            36 10565941.53    380373895    8715789   12238900 1120428.36 2023-06-01 2026-05-01
        의복            36  5849736.86    210590527    4371774    7341334  860882.76 2023-06-01 2026-05-01
       승용차            36  5573663.64    200651891    4236283    7032364  628419.43 2023-06-01 2026-05-01
      차량연료            36  4970993.81    178955777    43

2026-07-27 21:46:20,489 - INFO - 데이터 수집 완료 (테이블: DT_1KE10041): 2808 rows
2026-07-27 21:46:20,490 - INFO - 컬럼: ['C1_OBJ_NM', 'C2_NM', 'DT', 'C2', 'C1', 'PRD_SE', 'UNIT_NM_ENG', 'ITM_ID', 'TBL_ID', 'ITM_NM', 'TBL_NM', 'PRD_DE', 'LST_CHN_DE', 'C1_NM_ENG', 'C1_NM', 'UNIT_NM', 'ITM_NM_ENG', 'C2_OBJ_NM_ENG', 'C2_NM_ENG', 'ORG_ID', 'C1_OBJ_NM_ENG', 'C2_OBJ_NM']
2026-07-27 21:46:20,495 - INFO - 데이터 전처리 시작 (타입: online)
2026-07-27 21:46:20,515 - INFO - 결측치 확인:
C1_OBJ_NM         0
C2_NM             0
DT                0
C2                0
C1                0
PRD_SE            0
UNIT_NM_ENG       0
ITM_ID            0
TBL_ID            0
ITM_NM            0
TBL_NM            0
PRD_DE            0
LST_CHN_DE        0
C1_NM_ENG         0
C1_NM             0
UNIT_NM           0
ITM_NM_ENG        0
C2_OBJ_NM_ENG     0
C2_NM_ENG         0
ORG_ID            0
C1_OBJ_NM_ENG     0
C2_OBJ_NM         0
source_table      0
data_type         0
date              0
year              0
month             0
year_

   category  record_count   avg_sales  total_sales  min_sales  max_sales  std_sales first_date  last_date
         합계           108 14896131.61   1608782214    8346155   25543432 5488593.02 2023-06-01 2026-05-01
      음식서비스           108  2156037.06    232852002          0    3839553 1556621.77 2023-06-01 2026-05-01
      음·식료품           108  1999524.56    215948652     316852    3636364 1191381.19 2023-06-01 2026-05-01
 여행 및 교통서비스           108  1863415.17    201248838     121131    3277257 1214978.01 2023-06-01 2026-05-01
 가전·전자·통신기기           108  1271425.26    137313928     159808    2242439  716588.24 2023-06-01 2026-05-01
         의복           108  1261874.17    136282410     486958    2621099  545739.92 2023-06-01 2026-05-01
       생활용품           108  1083573.89    117025980      99117    1880121  694743.08 2023-06-01 2026-05-01
      가전·전자           108   906277.59     97877980     100635    1669475  539573.14 2023-06-01 2026-05-01
      농축수산물           108   755313.43     8157

2026-07-27 21:46:26,936 - INFO - Excel 파일 저장 완료: C:\Users\82108\OneDrive\INVESTMENT\지표상회_복제\result_files\KOSIS\kosis_retail_sales_20260727.xlsx
2026-07-27 21:46:26,937 - INFO - === KOSIS 소매판매 전체 데이터 수집 완료 ===



=== 판매채널별 YoY 성장률 (최근 6개월) ===
category_c1_nm   대형마트    면세점  무점포 소매    백화점  슈퍼마켓 및 잡화점  승용차 및 연료 소매점  전문소매점  \
date                                                                           
2025-12-01      -5.22 -11.00    5.80   4.23       -3.20          8.53   3.50   
2026-01-01     -18.09  12.18    8.53   9.99      -11.95          9.56   1.28   
2026-02-01      17.80  -3.82    2.86  18.17       16.00         -2.48   5.98   
2026-03-01     -11.91  -0.19   10.62  10.04       -0.79         13.25   8.15   
2026-04-01      -5.08  -5.54    6.16  15.92       -0.64          6.11   4.77   
2026-05-01      -7.28   5.62    9.63  19.29        0.04          3.41   5.50   

category_c1_nm   편의점  
date                  
2025-12-01      0.98  
2026-01-01      0.49  
2026-02-01      3.77  
2026-03-01      2.46  
2026-04-01      2.98  
2026-05-01      5.64  

=== 제화별 YoY 성장률 (최근 6개월) ===
category_c1_nm    가구   가전제품  기타내구재  기타비내구재  기타준내구재    내구재   비내구재  서적 문구  \
date                                  

In [4]:
# ===== 4. 저장 결과 확인 =====
for f in sorted(SAVE_DIR.glob("*.xlsx")):
    print(f.name, f"{f.stat().st_size/1024:.0f} KB")


kosis_retail_sales_20260727.xlsx 1394 KB
